In [ ]:
import json
from pathlib import Path
from scipy.integrate import trapezoid

from battledim_stage1 import (
    BattLeDIMStage1,
    Stage1Config,
    build_known_intervals_from_leak_columns,
    evaluate_stage1_results,
    load_battledim_merged_csv,
)


def resolve_existing_path(*candidates):
    seen = []
    for cand in candidates:
        if cand is None:
            continue
        p = Path(cand).expanduser()
        seen.append(str(p))
        if p.exists():
            return p
        if not p.is_absolute():
            cwd_p = Path.cwd() / p
            if cwd_p.exists():
                return cwd_p
            mnt_p = Path('/mnt/data') / p.name
            if mnt_p.exists():
                return mnt_p
    raise FileNotFoundError(f'Could not resolve any of: {seen}')


# One merged CSV containing Timestamp, press_*, flow_*, T1, n*, and labels.
DATA_PATH = str(resolve_existing_path(
    globals().get('DATA_PATH'),
    'data_full_2018_2019_fixed.csv',
    'BattleDIM/data_full_2018_2019_fixed.csv',
    '/mnt/data/data_full_2018_2019_fixed.csv',
))
CONFIG_PATH = str(resolve_existing_path(
    globals().get('CONFIG_PATH'),
    'battledim_stage1_tuned_config.json',
    '/mnt/data/battledim_stage1_tuned_config.json',
))

# Load config from JSON
with open(CONFIG_PATH, 'r') as f:
    cfg = Stage1Config(**json.load(f))

# Load and auto-detect BattLeDIM column groups from names.
df = load_battledim_merged_csv(DATA_PATH, timestamp_col='Timestamp')

# Standard BattLeDIM split: 2018 for fitting/calibration, 2019 for evaluation.
train_df = df[df.index.year == 2018].copy()
test_df = df[df.index.year == 2019].copy()

# Exclude labeled leak windows from training.
# leak_count / leak_mag_total / pipe_*_bin are preferred over leak_binary.
train_exclusions = build_known_intervals_from_leak_columns(
    train_df,
    pad_before='30min',
    pad_after='30min',
    min_active_samples=1,
)


# Recall-first defaults tuned for your merged schema.
# flow_3 is explicitly excluded from flow-based leak evidence because you noted it
# behaves like a tank-control square wave. It is still available elsewhere in the dataframe.
# uncooment if you don't want tp load config json
# cfg = Stage1Config(
#     flow_score_exclude_cols=['flow_3'],
#     burst_start_quantile=0.99,
#     incipient_start_quantile=0.99,
#     hold_quantile=0.95,
#     flow_hold_quantile=0.94,
#     top_n_anchor_pressures=6,
#     pairwise_anchors_per_sensor=4,
#     sensor_vote_threshold=1.2,
#     start_sensor_votes=2,
#     hold_sensor_votes=2,
#     quiet_end_samples=18,
#     merge_gap_samples=6,
#     stage2_recheck_hours=4.0,
#     weight_pressure=1.0,
#     weight_pairwise=0.8,
#     weight_flow=0.4,
#     weight_night_flow=0.25,
#     weight_slope=0.6,
# )

detector = BattLeDIMStage1(cfg)
detector.fit(train_df, known_intervals=train_exclusions)
results, sensor_detail, events = detector.detect(test_df)
triggers = detector.build_stage2_triggers(results)

# Optional evaluation against BattLeDIM labels in the merged table.
truth_any_leak = (test_df['leak_count'].fillna(0.0) > 0.0)
metrics = evaluate_stage1_results(results, truth_any_leak, triggers=triggers)

print('Resolved DATA_PATH:', DATA_PATH)
print('Resolved CONFIG_PATH:', CONFIG_PATH)
print('\nStage 1 metrics')
print(metrics)
print('\nAuto-excluded flow evidence columns:', detector.auto_excluded_flow_score_cols_)
print('Flow columns used for leak evidence:', detector.flow_score_cols_)
print('\nDetected events')
print(events.head(10))
print('\nStage 2 trigger preview')
print(triggers.head(20))

results.to_csv('stage1_scores.csv')
sensor_detail.to_csv('stage1_sensor_detail.csv')
events.to_csv('stage1_events.csv', index=False)
triggers.to_csv('stage2_triggers.csv', index=False)
metrics.to_csv('stage1_metrics.csv', header=['value'])

## Plot-ready data and external checks
This block uses the actual output columns from `battledim_stage1`:
- predicted leak flag: `leak_state`
- Stage 1 score: `score_final`
- Stage 2 calls: `triggers['timestamp']`

It also computes external false-positive / false-negative masks without changing the main detector code.


In [ ]:
import numpy as np
import pandas as pd

# Build one plotting dataframe aligned on the test index.
plot_df = test_df.copy()
plot_df["truth_leak"] = (plot_df["leak_count"].fillna(0) > 0).astype(int)
plot_df["stage1_leak"] = results["leak_state"].astype(int)
plot_df["stage1_score"] = results["score_final"].astype(float)
plot_df["event_id_pred"] = results["event_id"].astype(int)

# Optional truth details if present.
if "leak_mag_total" in plot_df.columns:
    plot_df["leak_mag_total"] = plot_df["leak_mag_total"].fillna(0.0)
if "leak_count" in plot_df.columns:
    plot_df["leak_count"] = plot_df["leak_count"].fillna(0.0)

# Confusion masks at each timestep.
plot_df["tp"] = ((plot_df["truth_leak"] == 1) & (plot_df["stage1_leak"] == 1)).astype(int)
plot_df["tn"] = ((plot_df["truth_leak"] == 0) & (plot_df["stage1_leak"] == 0)).astype(int)
plot_df["fp"] = ((plot_df["truth_leak"] == 0) & (plot_df["stage1_leak"] == 1)).astype(int)
plot_df["fn"] = ((plot_df["truth_leak"] == 1) & (plot_df["stage1_leak"] == 0)).astype(int)

# Stage 2 call flags aligned to the time index.
plot_df["stage2_call"] = 0
if not triggers.empty:
    trigger_times = pd.to_datetime(triggers["timestamp"])
    trigger_times = trigger_times[trigger_times.isin(plot_df.index)]
    plot_df.loc[trigger_times, "stage2_call"] = 1

# External metrics that do not require touching the detector code.
negative_steps = int((plot_df["truth_leak"] == 0).sum())
false_positive_steps = int(plot_df["fp"].sum())
false_negative_steps = int(plot_df["fn"].sum())
external_false_positive_rate = (false_positive_steps / negative_steps) if negative_steps > 0 else np.nan

extra_metrics = pd.Series({
    "stage1_active_fraction_external": float(plot_df["stage1_leak"].mean()),
    "stage2_call_fraction_external": float(plot_df["stage2_call"].mean()),
    "stage2_calls_per_day_external": float(plot_df["stage2_call"].resample("D").sum().mean()),
    "false_positive_rate_external": external_false_positive_rate,
    "n_negative_steps": negative_steps,
    "n_false_positive_steps": false_positive_steps,
    "n_false_negative_steps": false_negative_steps,
})

print("results columns:")
print(results.columns.tolist())
print("\nextra metrics:")
print(extra_metrics)
if not triggers.empty:
    print("\ntrigger reason counts:")
    print(triggers["reason"].value_counts())

print(
    f"\nStage 1 is active on {100 * plot_df['stage1_leak'].mean():.2f}% of timesteps, "
    f"but Stage 2 is called on only {100 * plot_df['stage2_call'].mean():.2f}% of timesteps."
)


## Plot helpers
- `plot_stage1_window(...)` gives a useful window with raw signals, truth vs Stage 1 labels, false negatives / false positives, score, leak count, leak magnitude, and Stage 2 calls.
- `plot_detected_event(...)` automatically uses the top sensors from a detected event.


In [ ]:
import matplotlib.pyplot as plt


def _slice_time(df, start=None, end=None):
    if start is None and end is None:
        return df.copy()
    if start is None:
        return df.loc[:end].copy()
    if end is None:
        return df.loc[start:].copy()
    return df.loc[start:end].copy()


def plot_stage1_window(plot_df, triggers, start=None, end=None, sensor_cols=None, title=None):
    s = _slice_time(plot_df, start=start, end=end)
    if s.empty:
        raise ValueError("Selected window is empty.")

    if sensor_cols is None:
        sensor_cols = ["press_14"]
    sensor_cols = [c for c in sensor_cols if c in s.columns]
    if not sensor_cols:
        raise ValueError("No valid sensor columns were provided.")

    trigger_slice = triggers.copy()
    if not trigger_slice.empty:
        trigger_slice["timestamp"] = pd.to_datetime(trigger_slice["timestamp"])
        if start is not None:
            trigger_slice = trigger_slice[trigger_slice["timestamp"] >= pd.Timestamp(start)]
        if end is not None:
            trigger_slice = trigger_slice[trigger_slice["timestamp"] <= pd.Timestamp(end)]

    nrows = len(sensor_cols) + 3
    fig, axes = plt.subplots(nrows, 1, figsize=(16, 2.8 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    # Raw sensors
    for i, col in enumerate(sensor_cols):
        ax = axes[i]
        ax.plot(s.index, s[col], linewidth=1)
        ax.set_ylabel(col)
        ax.set_title(col)

    # Truth vs prediction + mismatch regions
    ax_flag = axes[len(sensor_cols)]
    ax_flag.step(s.index, s["truth_leak"], where="post", linewidth=1.5, label="real leak")
    ax_flag.step(s.index, s["stage1_leak"], where="post", linewidth=1.5, label="Stage 1 leak")
    ax_flag.fill_between(s.index, 0, 1, where=s["fn"].astype(bool), step="post", alpha=0.25, label="false negative")
    ax_flag.fill_between(s.index, 0, 1, where=s["fp"].astype(bool), step="post", alpha=0.25, label="false positive")
    ax_flag.set_ylim(-0.05, 1.05)
    ax_flag.set_ylabel("flags")
    ax_flag.legend(loc="upper right", ncol=4)
    ax_flag.set_title("Real vs Stage 1 leak labels")

    # Score + Stage 2 triggers
    ax_score = axes[len(sensor_cols) + 1]
    ax_score.plot(s.index, s["stage1_score"], linewidth=1, label="Stage 1 score")
    if not trigger_slice.empty:
        reason_to_marker = {
            "event_start": "Start",
            "periodic_recheck": "Recheck",
            "sensor_shift": "Shift",
        }
        for reason, block in trigger_slice.groupby("reason"):
            ax_score.vlines(block["timestamp"], ymin=s["stage1_score"].min(), ymax=s["stage1_score"].max(), linewidth=0.8, alpha=0.7, label=reason_to_marker.get(reason, reason))
    ax_score.set_ylabel("score")
    ax_score.set_title("Stage 1 score and Stage 2 trigger times")
    ax_score.legend(loc="upper right")

    # Leak details + stage2 calls
    ax_det = axes[len(sensor_cols) + 2]
    ax_det.step(s.index, s["stage2_call"], where="post", linewidth=1.2, label="Stage 2 call")
    if "leak_count" in s.columns:
        ax_det.plot(s.index, s["leak_count"], linewidth=1.0, label="real leak_count")
    if "leak_mag_total" in s.columns:
        ax_det.plot(s.index, s["leak_mag_total"], linewidth=1.0, label="real leak_mag_total")
    ax_det.set_ylabel("detail")
    ax_det.set_title("Real leak detail and Stage 2 calls")
    ax_det.legend(loc="upper right")
    ax_det.set_xlabel("time")

    if title is None:
        title = f"Stage 1 window: {s.index.min()} to {s.index.max()}"
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_detected_event(plot_df, triggers, events, event_id=1, pad="2D", max_sensors=3):
    row = events.loc[events["event_id"] == event_id]
    if row.empty:
        raise ValueError(f"event_id {event_id} was not found.")
    row = row.iloc[0]
    start = pd.Timestamp(row["start_time"]) - pd.Timedelta(pad)
    end = pd.Timestamp(row["end_time"]) + pd.Timedelta(pad)
    sensors = [s.strip() for s in str(row["top_sensors"]).split(",") if s.strip()][:max_sensors]
    plot_stage1_window(
        plot_df,
        triggers,
        start=start,
        end=end,
        sensor_cols=sensors,
        title=f"Detected event {event_id} | sensors: {', '.join(sensors)}",
    )


## Example plots


In [ ]:
# 1) Detection lock-on window: shows the missed period before Stage 1 became active.
plot_stage1_window(
    plot_df,
    triggers,
    start="2019-01-20",
    end="2019-02-10",
    sensor_cols=["press_14", "press_22", "press_5"],
    title="Lock-on period around first detected leak",
)

# 2) Peak-period view around the strongest score.
plot_stage1_window(
    plot_df,
    triggers,
    start="2019-07-10",
    end="2019-07-30",
    sensor_cols=["press_1", "press_14", "press_27"],
    title="Peak score period",
)

# 3) Automatically use the top sensors from the detected event summary.
plot_detected_event(plot_df, triggers, events, event_id=1, pad="3D", max_sensors=3)


## Stage 2 pipeline

This section adds a realistic BattLeDIM Stage 2 workflow on top of the existing Stage 1 API:

- Stage 1 is re-fit on an early healthy 2018 portion only.
- Late 2018 is split chronologically into Stage 2 train / calibration / validation.
- 2019 remains the final holdout / inference period.
- Every important output is written to a timestamped run folder for reporting:
  - `tables/*.csv` and `tables/*.md`
  - `plots/*.png`
  - `run_summary.md`
  - `run_metadata.json`
  - `artifact_manifest.csv`

In [ ]:
import json
import math
import random
import warnings
from dataclasses import asdict, dataclass, is_dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42


def set_global_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_global_seed(SEED)


def resolve_existing_path(*candidates):
    seen = []
    for cand in candidates:
        if cand is None:
            continue
        p = Path(cand).expanduser()
        seen.append(str(p))
        if p.exists():
            return p
        if not p.is_absolute():
            cwd_p = Path.cwd() / p
            if cwd_p.exists():
                return cwd_p
            mnt_p = Path('/mnt/data') / p.name
            if mnt_p.exists():
                return mnt_p
    raise FileNotFoundError(f'Could not resolve any of: {seen}')


@dataclass
class Stage2RunConfig:
    data_path: str = 'data_full_2018_2019_fixed.csv'
    config_path: str = 'battledim_stage1_tuned_config.json'
    timestamp_col: str = 'Timestamp'
    lookback_steps: int = 144
    step_minutes: int = 5
    summary_horizons: Tuple[int, ...] = (12, 36, 72, 144)
    stage1_fit_healthy_frac: float = 0.60
    include_flow3_context: bool = True
    enable_regression_head: bool = True
    binary_min_recall: float = 0.95
    binary_precision_targets: Tuple[float, ...] = (0.50, 0.60, 0.70, 0.80, 0.90)
    easy_negative_ratio_per_positive: float = 0.35
    hard_peak_negative_ratio_per_positive: float = 0.25
    sampled_negative_weight: float = 0.55
    pseudo_hard_negative_weight: float = 0.90
    real_hard_negative_weight: float = 1.25
    output_root: str = 'battledim_stage2_runs'
    random_state: int = 42


PIPELINE_CFG = Stage2RunConfig(
    data_path=str(resolve_existing_path(globals().get('DATA_PATH'), 'data_full_2018_2019_fixed.csv', '/mnt/data/data_full_2018_2019_fixed.csv')),
    config_path=str(resolve_existing_path(globals().get('CONFIG_PATH'), 'battledim_stage1_tuned_config.json', '/mnt/data/battledim_stage1_tuned_config.json')),
)

RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path(PIPELINE_CFG.output_root) / f'run_{RUN_STAMP}'
TABLE_DIR = RUN_DIR / 'tables'
PLOT_DIR = RUN_DIR / 'plots'
for path in [RUN_DIR, TABLE_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

ARTIFACT_ROWS: List[Dict[str, str]] = []
REPORT_LINES: List[str] = []


def _json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp,)):
        return obj.isoformat()
    if isinstance(obj, (pd.Series,)):
        return obj.to_dict()
    if isinstance(obj, (pd.DataFrame,)):
        return obj.to_dict(orient='records')
    if is_dataclass(obj):
        return asdict(obj)
    return str(obj)


def _frame_to_markdown(df: pd.DataFrame, index: bool = False, max_rows: int = 50) -> str:
    block = df.copy()
    if max_rows is not None and len(block) > max_rows:
        block = block.head(max_rows)
    try:
        return block.to_markdown(index=index)
    except Exception:
        return '```\n' + block.to_string(index=index) + '\n```'


def add_artifact(kind: str, name: str, path: Path, description: str = '') -> None:
    ARTIFACT_ROWS.append(
        {
            'kind': kind,
            'name': name,
            'path': str(path.relative_to(RUN_DIR)),
            'description': description,
        }
    )


def save_json(obj, name: str, description: str = '') -> Path:
    path = RUN_DIR / f'{name}.json'
    path.write_text(json.dumps(obj, indent=2, default=_json_default))
    add_artifact('json', name, path, description)
    return path


def save_text(text: str, name: str, description: str = '') -> Path:
    path = RUN_DIR / name
    path.write_text(text)
    add_artifact('text', Path(name).stem, path, description)
    return path


def save_frame(df: pd.DataFrame, name: str, *, index: bool = False, description: str = '', max_md_rows: int = 50) -> Path:
    csv_path = TABLE_DIR / f'{name}.csv'
    md_path = TABLE_DIR / f'{name}.md'
    df.to_csv(csv_path, index=index)
    md_path.write_text(_frame_to_markdown(df, index=index, max_rows=max_md_rows))
    add_artifact('table', name, csv_path, description)
    add_artifact('markdown', name + '_md', md_path, description)
    return csv_path


def save_series(series: pd.Series, name: str, description: str = '') -> Path:
    df = series.rename('value').reset_index().rename(columns={'index': 'metric'})
    return save_frame(df, name, index=False, description=description, max_md_rows=len(df))


def save_figure(fig, name: str, description: str = '') -> Path:
    path = PLOT_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    add_artifact('plot', name, path, description)
    plt.close(fig)
    return path


def add_report_section(title: str, body: str) -> None:
    REPORT_LINES.append(f'## {title}\n\n{body}'.strip())


def finalize_run_summary() -> None:
    manifest_df = pd.DataFrame(ARTIFACT_ROWS)
    if not manifest_df.empty:
        manifest_df = manifest_df.sort_values(['kind', 'name']).reset_index(drop=True)
        save_frame(manifest_df, 'artifact_manifest', index=False, description='All files produced by this notebook run.', max_md_rows=200)
    summary_text = '# BattLeDIM Stage 2 run summary\n\n' + '\n\n'.join(REPORT_LINES)
    save_text(summary_text, 'run_summary.md', description='Narrative summary of the run for report drafting.')


print(f'Run folder: {RUN_DIR.resolve()}')
print(f'Table folder: {TABLE_DIR.resolve()}')
print(f'Plot folder: {PLOT_DIR.resolve()}')

In [ ]:
def natural_key(text: str):
    import re
    return [int(tok) if tok.isdigit() else tok for tok in re.split(r'(\\d+)', str(text))]



def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default



def infer_column_groups(df: pd.DataFrame, include_flow3_context: bool = True) -> Dict[str, List[str]]:
    pressure_cols = sorted([c for c in df.columns if c.startswith('press_')], key=natural_key)
    pipe_cols = sorted([c for c in df.columns if c.startswith('pipe_') and c.endswith('_bin')], key=natural_key)
    demand_cols = sorted([c for c in df.columns if c.startswith('n') and c[1:].isdigit()], key=natural_key)
    flow_primary = [c for c in ['flow_1', 'flow_2'] if c in df.columns]
    flow_context = ['flow_3'] if include_flow3_context and 'flow_3' in df.columns else []
    tank_cols = [c for c in ['T1'] if c in df.columns]
    return {
        'pressure_cols': pressure_cols,
        'pipe_cols': pipe_cols,
        'demand_cols': demand_cols,
        'flow_primary_cols': flow_primary,
        'flow_context_cols': flow_context,
        'tank_cols': tank_cols,
    }



def get_any_real_leak_mask(df: pd.DataFrame, pipe_cols: Sequence[str]) -> pd.Series:
    parts = []
    if pipe_cols:
        parts.append(df[list(pipe_cols)].fillna(0).sum(axis=1) > 0)
    if 'leak_count' in df.columns:
        parts.append(df['leak_count'].fillna(0) > 0)
    if 'leak_mag_total' in df.columns:
        parts.append(df['leak_mag_total'].fillna(0) > 0)
    if not parts and 'leak_binary' in df.columns:
        parts.append(df['leak_binary'].fillna(0) > 0)
    if not parts:
        raise ValueError('No usable leak labels found.')
    return pd.concat(parts, axis=1).any(axis=1)



def find_initial_healthy_block(index: pd.DatetimeIndex, healthy_mask: pd.Series) -> pd.DatetimeIndex:
    arr = healthy_mask.reindex(index).fillna(False).to_numpy(dtype=bool)
    end_pos = len(arr)
    for i, val in enumerate(arr):
        if not val:
            end_pos = i
            break
    return index[:end_pos]



def choose_stage1_fit_end_from_healthy_block(df_2018: pd.DataFrame, fit_frac: float, pipe_cols: Sequence[str]) -> pd.Timestamp:
    healthy_mask = ~get_any_real_leak_mask(df_2018, pipe_cols)
    initial_block = find_initial_healthy_block(df_2018.index, healthy_mask)
    if len(initial_block) >= 20:
        pos = max(1, int(math.floor(len(initial_block) * fit_frac))) - 1
        return initial_block[pos]
    pos = max(1, int(math.floor(len(df_2018) * 0.20))) - 1
    return df_2018.index[pos]



def chronological_split_frame(df_in: pd.DataFrame, frac: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if df_in.empty:
        return df_in.copy(), df_in.copy()
    df_sorted = df_in.sort_values('timestamp').reset_index(drop=True)
    if len(df_sorted) == 1:
        return df_sorted.copy(), df_sorted.iloc[0:0].copy()
    cut = max(1, int(math.floor(len(df_sorted) * frac)))
    cut = min(cut, len(df_sorted) - 1)
    return df_sorted.iloc[:cut].copy(), df_sorted.iloc[cut:].copy()



def split_df_threeway(df_in: pd.DataFrame, fracs=(0.60, 0.20, 0.20)):
    df_sorted = df_in.sort_values('timestamp').reset_index(drop=True).copy()
    if df_sorted.empty:
        empty = df_sorted.iloc[0:0].copy()
        return empty, empty, empty
    n = len(df_sorted)
    if n == 1:
        empty = df_sorted.iloc[0:0].copy()
        return df_sorted.copy(), empty, empty
    if n == 2:
        return df_sorted.iloc[:1].copy(), df_sorted.iloc[1:2].copy(), df_sorted.iloc[0:0].copy()
    a = max(1, int(round(n * fracs[0])))
    b = max(1, int(round(n * fracs[1])))
    if a + b >= n:
        b = max(1, n - a - 1)
    c = n - a - b
    if c <= 0:
        c = 1
        if b > 1:
            b -= 1
        else:
            a = max(1, a - 1)
    return df_sorted.iloc[:a].copy(), df_sorted.iloc[a:a+b].copy(), df_sorted.iloc[a+b:].copy()



def split_small_temporal(df_in: pd.DataFrame):
    df_sorted = df_in.sort_values('timestamp').reset_index(drop=True).copy()
    n = len(df_sorted)
    empty = df_sorted.iloc[0:0].copy()
    if n == 0:
        return empty, empty, empty
    if n == 1:
        return empty, empty, df_sorted.copy()
    if n == 2:
        return empty, df_sorted.iloc[:1].copy(), df_sorted.iloc[1:].copy()
    parts = np.array_split(df_sorted.index.to_numpy(), 3)
    return df_sorted.loc[parts[0]].copy(), df_sorted.loc[parts[1]].copy(), df_sorted.loc[parts[2]].copy()



def split_index_threeway(index: pd.DatetimeIndex):
    idx = pd.DatetimeIndex(sorted(pd.to_datetime(index).unique()))
    if len(idx) == 0:
        empty = pd.DatetimeIndex([])
        return empty, empty, empty
    parts = np.array_split(np.arange(len(idx)), 3)
    return idx[parts[0]], idx[parts[1]], idx[parts[2]]



def stratified_sample_timestamps(index: pd.DatetimeIndex, n_samples: int, seed: int = 42) -> pd.DatetimeIndex:
    idx = pd.DatetimeIndex(sorted(pd.to_datetime(index).unique()))
    if n_samples <= 0 or len(idx) == 0:
        return pd.DatetimeIndex([])
    if n_samples >= len(idx):
        return idx
    df_idx = pd.DataFrame({'timestamp': idx})
    df_idx['month'] = df_idx['timestamp'].dt.month
    df_idx['hour'] = df_idx['timestamp'].dt.hour
    df_idx['stratum'] = df_idx['month'].astype(str) + '_' + df_idx['hour'].astype(str)
    rng = np.random.RandomState(seed)
    sampled_parts = []
    base_frac = n_samples / len(df_idx)
    for _, block in df_idx.groupby('stratum', sort=True):
        take = int(round(len(block) * base_frac))
        take = min(len(block), max(0, take))
        if take > 0:
            sampled_parts.append(block.sample(n=take, random_state=int(rng.randint(0, 10_000_000))))
    sampled = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else df_idx.iloc[0:0].copy()
    sampled = sampled.drop_duplicates('timestamp')
    if len(sampled) < n_samples:
        remaining = df_idx.loc[~df_idx['timestamp'].isin(sampled['timestamp'])].copy()
        need = min(len(remaining), n_samples - len(sampled))
        if need > 0:
            extra = remaining.sample(n=need, random_state=int(rng.randint(0, 10_000_000)))
            sampled = pd.concat([sampled, extra], ignore_index=True)
    elif len(sampled) > n_samples:
        sampled = sampled.sample(n=n_samples, random_state=seed)
    return pd.DatetimeIndex(sorted(pd.to_datetime(sampled['timestamp']).unique()))



def make_pseudo_trigger_frame(index: pd.DatetimeIndex, reason: str, kind: str) -> pd.DataFrame:
    out = pd.DataFrame({'timestamp': pd.to_datetime(index)})
    out = out.sort_values('timestamp').reset_index(drop=True)
    out['event_id'] = -1
    out['reason'] = reason
    out['area_guess'] = None
    out['top_sensors'] = ''
    out['sampled_negative'] = 1
    out['pseudo_negative_kind'] = kind
    return out



def summarize_array(arr: np.ndarray, step_minutes: int) -> Dict[str, float]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {k: np.nan for k in ['last', 'mean', 'std', 'min', 'max', 'median', 'q10', 'q90', 'slope', 'delta', 'auc']}
    x = np.arange(arr.size, dtype=float)
    slope = float(np.polyfit(x, arr, 1)[0]) if arr.size >= 2 else 0.0
    return {
        'last': float(arr[-1]),
        'mean': float(arr.mean()),
        'std': float(arr.std(ddof=0)),
        'min': float(arr.min()),
        'max': float(arr.max()),
        'median': float(np.median(arr)),
        'q10': float(np.quantile(arr, 0.10)),
        'q90': float(np.quantile(arr, 0.90)),
        'slope': slope,
        'delta': float(arr[-1] - arr[0]) if arr.size >= 2 else 0.0,
        'auc': float(trapezoid(arr, dx=step_minutes)),       # 'auc': float(np.trapz(arr, dx=step_minutes)),
    }



def entropy_and_hhi(values: np.ndarray) -> Tuple[float, float]:
    x = np.asarray(values, dtype=float)
    x = np.where(np.isfinite(x), np.maximum(x, 0.0), 0.0)
    s = x.sum()
    if s <= 0:
        return 0.0, 0.0
    p = x / s
    mask = p > 0
    ent = -float(np.sum(p[mask] * np.log(p[mask]))) if mask.any() else 0.0
    hhi = float(np.sum(p ** 2))
    return ent, hhi



def topk_sensor_names(values: np.ndarray, sensor_names: Sequence[str], k: int = 3) -> List[str]:
    values = np.asarray(values, dtype=float)
    values = np.where(np.isfinite(values), values, -np.inf)
    if values.size == 0:
        return []
    idx = np.argsort(values)[::-1][:min(k, values.size)]
    return [sensor_names[i] for i in idx if np.isfinite(values[i])]



def mean_jaccard_over_time(topk_sets: List[set], current_set: set) -> float:
    if not topk_sets:
        return 0.0
    vals = []
    for s in topk_sets:
        union = len(s | current_set)
        vals.append((len(s & current_set) / union) if union > 0 else 0.0)
    return float(np.mean(vals)) if vals else 0.0



def infer_sensor_detail_map(sensor_detail: pd.DataFrame, pressure_cols: Sequence[str]) -> Dict[str, str]:
    numeric_cols = [c for c in sensor_detail.columns if pd.api.types.is_numeric_dtype(sensor_detail[c])]
    mapping = {}
    for sensor in pressure_cols:
        candidates = [
            c for c in numeric_cols
            if c == sensor or c.endswith(sensor) or c.startswith(sensor + '_') or (sensor in c)
        ]
        if candidates:
            candidates = sorted(set(candidates), key=lambda c: (c != sensor, len(c), c))
            mapping[sensor] = candidates[0]
    if len(mapping) >= max(1, len(pressure_cols) // 2):
        return mapping
    direct = [c for c in numeric_cols if c in pressure_cols]
    return {c: c for c in direct}



def add_causal_event_context(results: pd.DataFrame, triggers: pd.DataFrame, step_minutes: int):
    out = results.copy()
    out['event_age_samples_so_far'] = 0
    out['event_age_hours_so_far'] = 0.0
    out['event_running_max_score_final'] = 0.0
    out['event_running_mean_score_final'] = 0.0

    leak_state = out.get('leak_state', pd.Series(0, index=out.index)).fillna(0).astype(int)
    event_id = out.get('event_id', pd.Series(0, index=out.index)).fillna(0).astype(int)
    valid = (leak_state > 0) & (event_id > 0)
    if valid.any():
        tmp = out.loc[valid, ['score_final']].copy()
        tmp['event_id'] = event_id.loc[valid].values
        age = tmp.groupby('event_id').cumcount() + 1
        run_max = tmp.groupby('event_id')['score_final'].cummax()
        run_sum = tmp.groupby('event_id')['score_final'].cumsum()
        run_mean = run_sum / age
        out.loc[valid, 'event_age_samples_so_far'] = age.astype(int).values
        out.loc[valid, 'event_age_hours_so_far'] = age.astype(float).values * (step_minutes / 60.0)
        out.loc[valid, 'event_running_max_score_final'] = run_max.astype(float).values
        out.loc[valid, 'event_running_mean_score_final'] = run_mean.astype(float).values

    trg = triggers.copy()
    if not trg.empty:
        trg['timestamp'] = pd.to_datetime(trg['timestamp'])
        trg = trg.sort_values('timestamp').reset_index(drop=True)
    else:
        trg = pd.DataFrame(columns=['timestamp', 'event_id', 'reason', 'area_guess', 'top_sensors'])
    if 'sampled_negative' not in trg.columns:
        trg['sampled_negative'] = 0
    if 'pseudo_negative_kind' not in trg.columns:
        trg['pseudo_negative_kind'] = 'none'
    if 'event_id' not in trg.columns:
        trg['event_id'] = -1
    trg['event_id'] = trg['event_id'].fillna(-1).astype(int)
    trg['prior_triggers_in_event'] = 0
    trg['trigger_order_in_event'] = 0
    trg['minutes_since_prev_trigger_in_event'] = -1.0
    real = trg['event_id'] > 0
    if real.any():
        grp = trg.loc[real].groupby('event_id', sort=False)
        trg.loc[real, 'prior_triggers_in_event'] = grp.cumcount().values
        trg.loc[real, 'trigger_order_in_event'] = (grp.cumcount() + 1).values
        delta = grp['timestamp'].diff().dt.total_seconds().div(60.0).fillna(-1.0)
        trg.loc[real, 'minutes_since_prev_trigger_in_event'] = delta.values
    return out, trg


class Stage2FeatureBuilder:
    def __init__(self, raw_df: pd.DataFrame, results: pd.DataFrame, sensor_detail: pd.DataFrame, all_triggers: pd.DataFrame, groups: Dict[str, List[str]], config: Stage2RunConfig):
        self.raw_df = raw_df.copy()
        self.results, self.all_triggers = add_causal_event_context(results, all_triggers, config.step_minutes)
        self.sensor_detail = sensor_detail.copy()
        self.groups = groups
        self.config = config
        self.horizons = tuple(sorted(set(int(h) for h in config.summary_horizons if int(h) > 0 and int(h) <= config.lookback_steps)))
        self.index = self.raw_df.index
        self.pos_lookup = {ts: i for i, ts in enumerate(self.index)}
        self.any_leak_target = get_any_real_leak_mask(self.raw_df, self.groups['pipe_cols']).astype(int)
        self.stage1_numeric_cols = [
            c for c in [
                'pressure_topk', 'pairwise_topk', 'flow_topk', 'ewma_topk', 'slope_topk',
                'burst_raw', 'burst_cusum', 'score_burst_trigger', 'score_burst',
                'score_incipient', 'score_final', 'sensor_votes', 'leak_state'
            ] if c in self.results.columns
        ]
        self.top_sensor_cols = [c for c in [f'top_sensor_{i}' for i in range(1, 6)] if c in self.results.columns]
        self.sensor_detail_map = infer_sensor_detail_map(self.sensor_detail, self.groups['pressure_cols'])
        self.sensor_detail_sensor_names = [s for s in self.groups['pressure_cols'] if s in self.sensor_detail_map]
        self.sensor_detail_cols = [self.sensor_detail_map[s] for s in self.sensor_detail_sensor_names]

        reason_values = sorted(set(pd.Series(self.all_triggers.get('reason', pd.Series(dtype=object))).dropna().astype(str)))
        area_values = sorted(set(pd.Series(self.all_triggers.get('area_guess', pd.Series(dtype=object))).fillna('unknown').astype(str)))
        self.reason_values = reason_values or ['unknown']
        self.area_values = area_values or ['unknown']

        assert self.raw_df.index.equals(self.results.index)
        assert self.raw_df.index.equals(self.sensor_detail.index)

    def _window(self, ts: pd.Timestamp):
        pos = self.pos_lookup[pd.Timestamp(ts)]
        start = max(0, pos - self.config.lookback_steps + 1)
        return self.raw_df.iloc[start:pos+1], self.results.iloc[start:pos+1], self.sensor_detail.iloc[start:pos+1]

    def _add_horizon_features(self, feats: Dict[str, float], frame: pd.DataFrame, columns: Sequence[str], prefix: str) -> None:
        if not columns:
            return
        for col in columns:
            arr_full = frame[col].to_numpy(dtype=float, copy=False)
            for h in self.horizons:
                arr = arr_full[-min(h, len(arr_full)):]
                stats = summarize_array(arr, step_minutes=self.config.step_minutes)
                for stat_name, value in stats.items():
                    feats[f'{prefix}__{col}__h{h}__{stat_name}'] = value

    def _add_demand_features(self, feats: Dict[str, float], raw_win: pd.DataFrame) -> None:
        demand_cols = self.groups['demand_cols']
        if not demand_cols:
            return
        d = raw_win[demand_cols].fillna(0.0).to_numpy(dtype=float, copy=False)
        total = d.sum(axis=1)
        mean_node = d.mean(axis=1)
        std_node = d.std(axis=1)
        max_node = d.max(axis=1)
        median_node = np.median(d, axis=1)
        row_sum = total.reshape(-1, 1)
        share = np.divide(d, row_sum, out=np.zeros_like(d), where=row_sum > 0)
        hhi = (share ** 2).sum(axis=1)
        ent = np.zeros(share.shape[0], dtype=float)
        mask = share > 0
        if mask.any():
            ent = -(np.where(mask, share * np.log(np.where(mask, share, 1.0)), 0.0)).sum(axis=1)
        derived = {
            'total': total,
            'mean_node': mean_node,
            'std_node': std_node,
            'max_node': max_node,
            'median_node': median_node,
            'hhi': hhi,
            'entropy': ent,
        }
        for name, arr_full in derived.items():
            for h in self.horizons:
                arr = arr_full[-min(h, len(arr_full)):]
                stats = summarize_array(arr, step_minutes=self.config.step_minutes)
                for stat_name, value in stats.items():
                    feats[f'demand__{name}__h{h}__{stat_name}'] = value
        last = d[-1]
        feats['demand__last_cross__mean'] = float(last.mean())
        feats['demand__last_cross__std'] = float(last.std(ddof=0))
        feats['demand__last_cross__max'] = float(last.max())
        feats['demand__last_cross__median'] = float(np.median(last))
        feats['demand__last_cross__q90'] = float(np.quantile(last, 0.90))

    def _add_sensor_detail_features(self, feats: Dict[str, float], sensor_win: pd.DataFrame, result_win: pd.DataFrame) -> None:
        if not self.sensor_detail_cols:
            return
        sensor_frame = sensor_win[self.sensor_detail_cols]
        for sensor_name, col in zip(self.sensor_detail_sensor_names, self.sensor_detail_cols):
            arr_full = sensor_frame[col].to_numpy(dtype=float, copy=False)
            for h in self.horizons:
                arr = arr_full[-min(h, len(arr_full)):]
                stats = summarize_array(arr, step_minutes=self.config.step_minutes)
                for stat_name in ['last', 'mean', 'max', 'slope']:
                    feats[f'sensor_detail__{sensor_name}__h{h}__{stat_name}'] = stats[stat_name]

        sensor_values = sensor_frame.to_numpy(dtype=float, copy=False)
        current_vals = np.where(np.isfinite(sensor_values[-1]), sensor_values[-1], 0.0)
        sorted_vals = np.sort(current_vals)[::-1]
        ent, hhi = entropy_and_hhi(current_vals)
        feats['sensor_detail_summary__current_max'] = float(sorted_vals[0]) if sorted_vals.size > 0 else 0.0
        feats['sensor_detail_summary__current_mean'] = float(current_vals.mean()) if current_vals.size > 0 else 0.0
        feats['sensor_detail_summary__current_std'] = float(current_vals.std(ddof=0)) if current_vals.size > 0 else 0.0
        feats['sensor_detail_summary__current_top1'] = float(sorted_vals[0]) if sorted_vals.size > 0 else 0.0
        feats['sensor_detail_summary__current_top2'] = float(sorted_vals[1]) if sorted_vals.size > 1 else 0.0
        feats['sensor_detail_summary__current_top3'] = float(sorted_vals[2]) if sorted_vals.size > 2 else 0.0
        feats['sensor_detail_summary__current_gap12'] = float(sorted_vals[0] - sorted_vals[1]) if sorted_vals.size > 1 else 0.0
        feats['sensor_detail_summary__current_entropy'] = ent
        feats['sensor_detail_summary__current_hhi'] = hhi
        feats['sensor_detail_summary__current_top1_frac'] = float(sorted_vals[0] / current_vals.sum()) if current_vals.sum() > 0 else 0.0

        rank_order = np.argsort(current_vals)[::-1]
        rank_map = {self.sensor_detail_sensor_names[idx]: rank + 1 for rank, idx in enumerate(rank_order)}
        current_top_sensor = self.sensor_detail_sensor_names[rank_order[0]] if len(rank_order) else None
        topk_sets = []
        top1_names = []
        for row_vals in sensor_values:
            vals = np.where(np.isfinite(row_vals), row_vals, 0.0)
            top1 = topk_sensor_names(vals, self.sensor_detail_sensor_names, k=1)
            top3 = topk_sensor_names(vals, self.sensor_detail_sensor_names, k=3)
            top1_names.append(top1[0] if top1 else '')
            topk_sets.append(set(top3))
        current_top3 = set(topk_sensor_names(current_vals, self.sensor_detail_sensor_names, k=3))
        feats['sensor_detail_summary__top1_persistence'] = float(np.mean([name == current_top_sensor for name in top1_names])) if current_top_sensor else 0.0
        feats['sensor_detail_summary__top3_jaccard_mean'] = mean_jaccard_over_time(topk_sets, current_top3)

        for i, col in enumerate(self.top_sensor_cols, start=1):
            sensor = str(result_win.iloc[-1][col]) if col in result_win.columns else ''
            sensor = sensor if sensor in rank_map else ''
            feats[f'sensor_detail_summary__rank_of_result_top_sensor_{i}'] = float(rank_map.get(sensor, len(self.sensor_detail_sensor_names) + 1))

    def _add_top_sensor_context(self, feats: Dict[str, float], result_win: pd.DataFrame) -> None:
        if not self.top_sensor_cols:
            return
        total_rows = max(1, len(result_win))
        weighted_counts = {sensor: 0.0 for sensor in self.groups['pressure_cols']}
        raw_counts = {sensor: 0.0 for sensor in self.groups['pressure_cols']}
        current_weight = {sensor: 0.0 for sensor in self.groups['pressure_cols']}
        current_rank = {sensor: 0.0 for sensor in self.groups['pressure_cols']}
        for rank, col in enumerate(self.top_sensor_cols, start=1):
            series = result_win[col].fillna('').astype(str)
            for sensor, count in series.value_counts().to_dict().items():
                if sensor in raw_counts:
                    raw_counts[sensor] += float(count)
                    weighted_counts[sensor] += float(count) / rank
            current_sensor = str(result_win.iloc[-1][col])
            if current_sensor in current_weight:
                current_weight[current_sensor] += 1.0 / rank
                current_rank[current_sensor] = float(rank)
        for sensor in self.groups['pressure_cols']:
            feats[f'top_sensor_history_frac__{sensor}'] = raw_counts[sensor] / total_rows
            feats[f'top_sensor_history_weighted_frac__{sensor}'] = weighted_counts[sensor] / total_rows
            feats[f'top_sensor_current_weight__{sensor}'] = current_weight[sensor]
            feats[f'top_sensor_current_rank__{sensor}'] = current_rank[sensor]

    def _add_trigger_metadata(self, feats: Dict[str, float], trigger_row: pd.Series) -> None:
        reason = str(trigger_row.get('reason', 'unknown'))
        area = str(trigger_row.get('area_guess', 'unknown'))
        area = area if area and area != 'nan' else 'unknown'
        for val in self.reason_values:
            feats[f'trigger__reason__{val}'] = 1.0 if reason == val else 0.0
        for val in self.area_values:
            feats[f'trigger__area_guess__{val}'] = 1.0 if area == val else 0.0
        feats['trigger__is_sampled_negative'] = float(int(trigger_row.get('sampled_negative', 0)))
        feats['trigger__prior_triggers_in_event'] = safe_float(trigger_row.get('prior_triggers_in_event', 0), 0.0)
        feats['trigger__trigger_order_in_event'] = safe_float(trigger_row.get('trigger_order_in_event', 0), 0.0)
        feats['trigger__minutes_since_prev_trigger_in_event'] = safe_float(trigger_row.get('minutes_since_prev_trigger_in_event', -1.0), -1.0)

    def _add_calendar_features(self, feats: Dict[str, float], ts: pd.Timestamp) -> None:
        hour = ts.hour + ts.minute / 60.0
        dow = ts.dayofweek
        month = ts.month
        feats['calendar__hour_sin'] = math.sin(2 * math.pi * hour / 24.0)
        feats['calendar__hour_cos'] = math.cos(2 * math.pi * hour / 24.0)
        feats['calendar__dow_sin'] = math.sin(2 * math.pi * dow / 7.0)
        feats['calendar__dow_cos'] = math.cos(2 * math.pi * dow / 7.0)
        feats['calendar__month_sin'] = math.sin(2 * math.pi * month / 12.0)
        feats['calendar__month_cos'] = math.cos(2 * math.pi * month / 12.0)
        feats['calendar__is_night'] = float(1 if ts.hour in [0, 1, 2, 3, 4, 5] else 0)

    def build_dataset(self, trigger_df: pd.DataFrame, split_name: str):
        if trigger_df.empty:
            return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
        features, metas, targets = [], [], []
        trigger_df = trigger_df.copy()
        trigger_df['timestamp'] = pd.to_datetime(trigger_df['timestamp'])
        trigger_df = trigger_df.sort_values('timestamp').reset_index(drop=True)
        for _, trigger_row in trigger_df.iterrows():
            ts = pd.Timestamp(trigger_row['timestamp'])
            if ts not in self.pos_lookup:
                continue
            raw_win, result_win, sensor_win = self._window(ts)
            cur_raw = self.raw_df.loc[ts]
            cur_result = self.results.loc[ts]
            feats: Dict[str, float] = {}
            self._add_calendar_features(feats, ts)
            self._add_trigger_metadata(feats, trigger_row)
            raw_cols = self.groups['pressure_cols'] + self.groups['flow_primary_cols'] + self.groups['flow_context_cols'] + self.groups['tank_cols']
            self._add_horizon_features(feats, raw_win, raw_cols, prefix='raw')
            self._add_demand_features(feats, raw_win)
            self._add_horizon_features(feats, result_win, self.stage1_numeric_cols, prefix='stage1')
            feats['event_ctx__event_age_samples_so_far'] = safe_float(cur_result.get('event_age_samples_so_far', 0), 0.0)
            feats['event_ctx__event_age_hours_so_far'] = safe_float(cur_result.get('event_age_hours_so_far', 0.0), 0.0)
            feats['event_ctx__event_running_max_score_final'] = safe_float(cur_result.get('event_running_max_score_final', 0.0), 0.0)
            feats['event_ctx__event_running_mean_score_final'] = safe_float(cur_result.get('event_running_mean_score_final', 0.0), 0.0)
            self._add_sensor_detail_features(feats, sensor_win, result_win)
            self._add_top_sensor_context(feats, result_win)
            features.append(feats)

            y_any = int(self.any_leak_target.loc[ts])
            y_pipe = cur_raw[self.groups['pipe_cols']].fillna(0).astype(int).to_dict()
            target_row = {'y_any_leak': y_any}
            target_row.update(y_pipe)
            if 'leak_mag_total' in cur_raw.index:
                target_row['y_leak_mag_total'] = float(cur_raw['leak_mag_total'])
            if 'leak_count' in cur_raw.index:
                target_row['y_leak_count'] = float(cur_raw['leak_count'])
            targets.append(target_row)

            metas.append({
                'split': split_name,
                'timestamp': ts,
                'event_id': int(safe_float(trigger_row.get('event_id', -1), -1)),
                'reason': str(trigger_row.get('reason', 'unknown')),
                'area_guess': str(trigger_row.get('area_guess', 'unknown')),
                'sampled_negative': int(trigger_row.get('sampled_negative', 0)),
                'pseudo_negative_kind': str(trigger_row.get('pseudo_negative_kind', 'none')),
                'top_sensors': str(trigger_row.get('top_sensors', '')),
            })
        X = pd.DataFrame(features).replace([np.inf, -np.inf], np.nan)
        meta = pd.DataFrame(metas)
        y = pd.DataFrame(targets)
        return X, meta, y

In [ ]:
class ConstantBinaryModel:
    def __init__(self, p: float):
        self.p = float(np.clip(p, 1e-6, 1 - 1e-6))
        self.feature_importances_ = None

    def fit(self, X, y, sample_weight=None):
        return self

    def predict_proba(self, X):
        p = np.full(len(X), self.p, dtype=float)
        return np.column_stack([1.0 - p, p])


class ProbabilityCalibrator:
    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.method = 'identity'
        self.model = None

    def fit(self, raw_p: np.ndarray, y_true: np.ndarray):
        raw_p = np.clip(np.asarray(raw_p, dtype=float), 1e-6, 1 - 1e-6)
        y_true = np.asarray(y_true, dtype=int)
        classes, counts = np.unique(y_true, return_counts=True)
        if len(classes) < 2:
            self.method = 'identity'
            self.model = None
            return self
        if counts.min() >= 20:
            self.method = 'isotonic'
            self.model = IsotonicRegression(out_of_bounds='clip')
            self.model.fit(raw_p, y_true)
            return self
        self.method = 'platt'
        z = np.log(raw_p / (1.0 - raw_p)).reshape(-1, 1)
        self.model = LogisticRegression(random_state=self.random_state, max_iter=1000)
        self.model.fit(z, y_true)
        return self

    def predict(self, raw_p: np.ndarray) -> np.ndarray:
        raw_p = np.clip(np.asarray(raw_p, dtype=float), 1e-6, 1 - 1e-6)
        if self.model is None:
            return raw_p
        if self.method == 'isotonic':
            return np.clip(self.model.predict(raw_p), 1e-6, 1 - 1e-6)
        z = np.log(raw_p / (1.0 - raw_p)).reshape(-1, 1)
        return np.clip(self.model.predict_proba(z)[:, 1], 1e-6, 1 - 1e-6)



def make_binary_estimator(random_state: int = 42):
    try:
        import lightgbm as lgb
        model = lgb.LGBMClassifier(
            objective='binary',
            n_estimators=400,
            learning_rate=0.04,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
        return model, 'lightgbm'
    except Exception:
        try:
            from xgboost import XGBClassifier
            model = XGBClassifier(
                objective='binary:logistic',
                n_estimators=400,
                learning_rate=0.04,
                max_depth=6,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                eval_metric='logloss',
                random_state=random_state,
                n_jobs=-1,
            )
            return model, 'xgboost'
        except Exception:
            from sklearn.ensemble import HistGradientBoostingClassifier
            model = HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_depth=6,
                max_iter=400,
                random_state=random_state,
            )
            return model, 'sklearn_hgb'



def make_regressor_estimator(random_state: int = 42):
    try:
        import lightgbm as lgb
        base = lgb.LGBMRegressor(
            objective='regression',
            n_estimators=400,
            learning_rate=0.04,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
        return MultiOutputRegressor(base), 'lightgbm'
    except Exception:
        try:
            from xgboost import XGBRegressor
            base = XGBRegressor(
                objective='reg:squarederror',
                n_estimators=400,
                learning_rate=0.04,
                max_depth=6,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                random_state=random_state,
                n_jobs=-1,
            )
            return MultiOutputRegressor(base), 'xgboost'
        except Exception:
            from sklearn.ensemble import HistGradientBoostingRegressor
            base = HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_depth=6,
                max_iter=400,
                random_state=random_state,
            )
            return MultiOutputRegressor(base), 'sklearn_hgb'



def fit_binary_model(X_train: pd.DataFrame, y_train: np.ndarray, sample_weight: Optional[np.ndarray], random_state: int = 42):
    classes = np.unique(y_train)
    if len(classes) < 2:
        model = ConstantBinaryModel(p=float(np.mean(y_train)))
        return model, 'constant'
    model, backend = make_binary_estimator(random_state=random_state)
    fit_kwargs = {}
    if sample_weight is not None:
        fit_kwargs['sample_weight'] = sample_weight
    try:
        model.fit(X_train, y_train, **fit_kwargs)
    except TypeError:
        model.fit(X_train, y_train)
    return model, backend



def fit_regression_model(X_train: pd.DataFrame, y_train: pd.DataFrame, sample_weight: Optional[np.ndarray], random_state: int = 42):
    model, backend = make_regressor_estimator(random_state=random_state)
    try:
        model.fit(X_train, y_train, sample_weight=sample_weight)
    except TypeError:
        model.fit(X_train, y_train)
    return model, backend



def select_peak_negative_timestamps(score: pd.Series, n_samples: int, step_minutes: int = 5, min_gap_steps: int = 12, peak_radius: int = 3) -> pd.DatetimeIndex:
    s = score.dropna().astype(float).sort_index()
    if n_samples <= 0 or s.empty:
        return pd.DatetimeIndex([])
    roll = s.rolling(2 * peak_radius + 1, center=True, min_periods=1).max()
    cand = s[s >= (roll - 1e-12)].sort_values(ascending=False)
    selected = []
    min_gap_seconds = step_minutes * 60 * min_gap_steps
    for ts, _ in cand.items():
        if len(selected) >= n_samples:
            break
        if any(abs((pd.Timestamp(ts) - prev).total_seconds()) < min_gap_seconds for prev in selected):
            continue
        selected.append(pd.Timestamp(ts))
    return pd.DatetimeIndex(sorted(selected))



def align_feature_frames(*frames):
    shared_cols = list(frames[0].columns)
    aligned = [f.reindex(columns=shared_cols).copy() for f in frames]
    keep_cols = aligned[0].columns[aligned[0].nunique(dropna=False) > 1].tolist()
    aligned = [f[keep_cols].copy() for f in aligned]
    med = aligned[0].median(numeric_only=True)
    aligned = [f.fillna(med).astype(np.float32) for f in aligned]
    return aligned



def direct_threshold_search(y_true, p, min_recall=0.95):
    y_true = np.asarray(y_true, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y_true) == 0:
        return {'threshold': 0.5, 'precision': np.nan, 'recall': np.nan, 'f1': np.nan}, pd.DataFrame()
    grid = np.unique(np.r_[0.0, np.linspace(0.0, 1.0, 501), np.clip(p, 0.0, 1.0)])
    rows = []
    for thr in grid:
        y_pred = (p >= thr).astype(int)
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        rows.append({'threshold': float(thr), 'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'accepted_fraction': float(y_pred.mean())})
    tbl = pd.DataFrame(rows).drop_duplicates('threshold').sort_values('threshold').reset_index(drop=True)
    eligible = tbl[tbl['recall'] >= min_recall]
    if not eligible.empty:
        best = eligible.sort_values(['precision', 'f1', 'threshold'], ascending=[False, False, False]).iloc[0]
    else:
        best = tbl.sort_values(['recall', 'precision', 'f1', 'threshold'], ascending=[False, False, False]).iloc[0]
    return {'threshold': float(best['threshold']), 'precision': float(best['precision']), 'recall': float(best['recall']), 'f1': float(best['f1'])}, tbl



def select_binary_feature_columns(columns):
    keep = []
    for c in columns:
        if c.startswith('calendar__'):
            continue
        if c.startswith('event_ctx__'):
            continue
        if c.startswith('trigger__reason__'):
            continue
        if c.startswith('trigger__area_guess__'):
            continue
        if c in {'trigger__is_sampled_negative', 'trigger__prior_triggers_in_event', 'trigger__trigger_order_in_event', 'trigger__minutes_since_prev_trigger_in_event'}:
            continue
        if c.startswith('stage1__leak_state__'):
            continue
        keep.append(c)
    return keep



def select_localisation_feature_columns(columns):
    keep = []
    for c in columns:
        if c == 'trigger__is_sampled_negative':
            continue
        if c.startswith('trigger__reason__sampled_negative'):
            continue
        if c.startswith('trigger__reason__pseudo_hard_negative'):
            continue
        if c.startswith('trigger__area_guess__sampled_negative'):
            continue
        keep.append(c)
    return keep



def safe_binary_metrics(y_true: np.ndarray, p: np.ndarray, threshold: float, precision_targets: Sequence[float]) -> pd.Series:
    y_true = np.asarray(y_true, dtype=int)
    p = np.asarray(p, dtype=float)
    if len(y_true) == 0:
        return pd.Series(dtype=float)
    y_pred = (p >= threshold).astype(int)
    out = {
        'n_packets': int(len(y_true)),
        'positive_rate': float(y_true.mean()) if len(y_true) else np.nan,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'accepted_fraction': float(y_pred.mean()) if len(y_pred) else np.nan,
        'brier': brier_score_loss(y_true, p) if len(y_true) else np.nan,
    }
    if len(np.unique(y_true)) >= 2:
        out['pr_auc'] = average_precision_score(y_true, p)
        out['roc_auc'] = roc_auc_score(y_true, p)
        for target in precision_targets:
            best_recall = np.nan
            for thr in np.unique(np.r_[0.0, np.linspace(0.0, 1.0, 501), np.clip(p, 0.0, 1.0)]):
                y_hat = (p >= thr).astype(int)
                prec = precision_score(y_true, y_hat, zero_division=0)
                rec = recall_score(y_true, y_hat, zero_division=0)
                if prec >= target:
                    if np.isnan(best_recall) or rec > best_recall:
                        best_recall = rec
            out[f'recall_at_precision_{target:.2f}'] = best_recall
    else:
        out['pr_auc'] = np.nan
        out['roc_auc'] = np.nan
        for target in precision_targets:
            out[f'recall_at_precision_{target:.2f}'] = np.nan
    return pd.Series(out)



def evaluate_localisation(y_true_pipe: pd.DataFrame, pipe_prob_df: pd.DataFrame, threshold: float = 0.50) -> pd.Series:
    if y_true_pipe.empty or pipe_prob_df.empty:
        return pd.Series(dtype=float)
    y_true = y_true_pipe.to_numpy(dtype=int)
    y_prob = pipe_prob_df[y_true_pipe.columns].to_numpy(dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    per_pipe_f1 = []
    per_pipe_ap = []
    for i, col in enumerate(y_true_pipe.columns):
        per_pipe_f1.append(f1_score(y_true[:, i], y_pred[:, i], zero_division=0))
        if len(np.unique(y_true[:, i])) >= 2:
            per_pipe_ap.append(average_precision_score(y_true[:, i], y_prob[:, i]))
    top1_idx = np.argmax(y_prob, axis=1)
    top3_idx = np.argsort(y_prob, axis=1)[:, ::-1][:, :3]
    positive_rows = y_true.sum(axis=1) > 0
    top1_hit = []
    top3_hit = []
    for i in range(len(y_true)):
        if not positive_rows[i]:
            continue
        top1_hit.append(float(y_true[i, top1_idx[i]] == 1))
        top3_hit.append(float(np.any(y_true[i, top3_idx[i]] == 1)))
    return pd.Series({
        'macro_f1_pipes': float(np.mean(per_pipe_f1)) if per_pipe_f1 else np.nan,
        'top1_hit_rate': float(np.mean(top1_hit)) if top1_hit else np.nan,
        'top3_hit_rate': float(np.mean(top3_hit)) if top3_hit else np.nan,
        'mAP_pipes': float(np.mean(per_pipe_ap)) if per_pipe_ap else np.nan,
    })



def packet_summary(meta, y, split_name):
    if meta.empty:
        return pd.Series({'split': split_name, 'n_packets': 0})
    actual = meta['sampled_negative'].astype(int) == 0
    pseudo_kind = meta.get('pseudo_negative_kind', pd.Series('', index=meta.index)).fillna('')
    return pd.Series({
        'split': split_name,
        'n_packets': int(len(meta)),
        'n_actual_triggers': int(actual.sum()),
        'n_actual_positive': int(((actual) & (y['y_any_leak'] == 1)).sum()),
        'n_actual_negative': int(((actual) & (y['y_any_leak'] == 0)).sum()),
        'n_pseudo_hard_negative': int((pseudo_kind == 'pseudo_hard_negative').sum()),
        'n_easy_negative': int((pseudo_kind == 'sampled_negative').sum()),
        'positive_rate': float(y['y_any_leak'].mean()) if len(y) else np.nan,
    })



def operational_binary_summary(meta: pd.DataFrame, y: pd.DataFrame, p: np.ndarray, threshold: float) -> pd.Series:
    out = meta.copy().reset_index(drop=True)
    out['y_any_leak'] = y['y_any_leak'].values
    out['pred_any_leak'] = (np.asarray(p) >= threshold).astype(int)
    out['actual_trigger'] = (out['sampled_negative'].astype(int) == 0).astype(int)
    out['real_hard_negative'] = ((out['actual_trigger'] == 1) & (out['y_any_leak'] == 0)).astype(int)
    return pd.Series({
        'n_packets': int(len(out)),
        'n_actual_triggers': int(out['actual_trigger'].sum()),
        'n_real_hard_negatives': int(out['real_hard_negative'].sum()),
        'accept_frac_all_packets': float(out['pred_any_leak'].mean()) if len(out) else np.nan,
        'accept_frac_actual_triggers': float(out.loc[out['actual_trigger'] == 1, 'pred_any_leak'].mean()) if int(out['actual_trigger'].sum()) > 0 else np.nan,
        'accept_frac_real_hard_negatives': float(out.loc[out['real_hard_negative'] == 1, 'pred_any_leak'].mean()) if int(out['real_hard_negative'].sum()) > 0 else np.nan,
    })



def reason_acceptance_breakdown(pred_df: pd.DataFrame) -> pd.DataFrame:
    if pred_df.empty:
        return pd.DataFrame()
    actual = pred_df[pred_df['sampled_negative'] == 0].copy()
    if actual.empty:
        return pd.DataFrame()
    out = (
        actual.assign(decision=np.where(actual['pred_any_leak'] == 1, 'accepted', 'rejected'))
        .groupby(['reason', 'decision'])
        .size()
        .unstack(fill_value=0)
    )
    if 'accepted' not in out.columns:
        out['accepted'] = 0
    if 'rejected' not in out.columns:
        out['rejected'] = 0
    out['total'] = out['accepted'] + out['rejected']
    out['accepted_frac'] = out['accepted'] / out['total'].replace(0, np.nan)
    out['rejected_frac'] = out['rejected'] / out['total'].replace(0, np.nan)
    return out.sort_values('total', ascending=False)



def build_prediction_frame(meta: pd.DataFrame, y: pd.DataFrame, p_raw: np.ndarray, p_cal: np.ndarray, threshold: float, pipe_prob_df: pd.DataFrame, reg_pred: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    out = meta.copy().reset_index(drop=True)
    out['y_any_leak'] = y['y_any_leak'].values
    out['p_any_leak_raw'] = p_raw
    out['p_any_leak'] = p_cal
    out['pred_any_leak'] = (p_cal >= threshold).astype(int)
    pipe_names = list(pipe_prob_df.columns)
    pipe_values = pipe_prob_df.to_numpy(dtype=float)
    if pipe_values.size:
        top_idx = np.argsort(pipe_values, axis=1)[:, ::-1][:, :3]
        out['top_1_pipe'] = [pipe_names[idx[0]].replace('_bin', '') for idx in top_idx]
        out['top_2_pipe'] = [pipe_names[idx[1]].replace('_bin', '') for idx in top_idx]
        out['top_3_pipe'] = [pipe_names[idx[2]].replace('_bin', '') for idx in top_idx]
        out['top_3_pipes'] = [', '.join([pipe_names[j].replace('_bin', '') for j in idx]) for idx in top_idx]
    else:
        out['top_1_pipe'] = ''
        out['top_2_pipe'] = ''
        out['top_3_pipe'] = ''
        out['top_3_pipes'] = ''
    true_pipe_cols = [c for c in y.columns if c.startswith('pipe_') and c.endswith('_bin')]
    out['true_pipes'] = [', '.join([c.replace('_bin', '') for c in true_pipe_cols if y.iloc[i][c] == 1]) for i in range(len(y))]
    if reg_pred is not None and not reg_pred.empty:
        for c in reg_pred.columns:
            out[c] = reg_pred[c].values
    return out.sort_values('timestamp').reset_index(drop=True)



def feature_group(name: str) -> str:
    if name.startswith('raw__press_'):
        return 'raw_pressure'
    if name.startswith('raw__flow_') or name.startswith('raw__T1'):
        return 'raw_flow_tank'
    if name.startswith('demand__'):
        return 'demand_agg'
    if name.startswith('stage1__'):
        return 'stage1_history'
    if name.startswith('sensor_detail__'):
        return 'sensor_detail_per_sensor'
    if name.startswith('sensor_detail_summary__'):
        return 'sensor_detail_summary'
    if name.startswith('top_sensor_'):
        return 'top_sensor_context'
    if name.startswith('trigger__'):
        return 'trigger_metadata'
    if name.startswith('event_ctx__'):
        return 'event_context'
    if name.startswith('calendar__'):
        return 'calendar'
    return 'other'



def feature_importance_frame(model, feature_names: Sequence[str], X_val: Optional[pd.DataFrame] = None, y_val: Optional[np.ndarray] = None, max_proxy_features: int = 60, n_repeats: int = 3) -> pd.DataFrame:
    if hasattr(model, 'feature_importances_') and getattr(model, 'feature_importances_', None) is not None:
        imp = np.asarray(model.feature_importances_, dtype=float)
        return pd.DataFrame({'feature': list(feature_names), 'importance': imp, 'importance_type': 'native'}).sort_values('importance', ascending=False)
    if X_val is None or y_val is None or len(X_val) == 0:
        return pd.DataFrame(columns=['feature', 'importance', 'importance_type'])

    y_val = np.asarray(y_val, dtype=int)
    assoc = []
    for c in X_val.columns:
        x = np.asarray(X_val[c], dtype=float)
        if np.nanstd(x) == 0 or len(np.unique(y_val)) < 2:
            assoc.append(0.0)
        else:
            corr = np.corrcoef(np.nan_to_num(x), y_val)[0, 1]
            assoc.append(abs(corr) if np.isfinite(corr) else 0.0)
    candidate_df = pd.DataFrame({'feature': X_val.columns, 'assoc': assoc})
    candidates = candidate_df.sort_values('assoc', ascending=False)['feature'].head(max_proxy_features).tolist()
    if not candidates:
        return pd.DataFrame(columns=['feature', 'importance', 'importance_type'])

    if len(np.unique(y_val)) < 2:
        return pd.DataFrame({'feature': candidates, 'importance': 0.0, 'importance_type': 'proxy_permutation'})

    base_p = model.predict_proba(X_val)[:, 1]
    base_score = average_precision_score(y_val, base_p)
    rng = np.random.RandomState(42)
    rows = []
    for c in candidates:
        drops = []
        for _ in range(n_repeats):
            X_perm = X_val.copy()
            X_perm[c] = rng.permutation(X_perm[c].to_numpy())
            p_perm = model.predict_proba(X_perm)[:, 1]
            score_perm = average_precision_score(y_val, p_perm)
            drops.append(base_score - score_perm)
        rows.append({'feature': c, 'importance': float(np.mean(drops)), 'importance_type': 'proxy_permutation'})
    return pd.DataFrame(rows).sort_values('importance', ascending=False)



def make_pr_curve_fig(pred_df: pd.DataFrame, title: str):
    if pred_df.empty or pred_df['y_any_leak'].nunique() < 2:
        return None
    from sklearn.metrics import precision_recall_curve
    precision, recall, _ = precision_recall_curve(pred_df['y_any_leak'], pred_df['p_any_leak'])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(recall, precision)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(title)
    fig.tight_layout()
    return fig



def make_calibration_fig(pred_df: pd.DataFrame, title: str):
    if pred_df.empty or pred_df['y_any_leak'].nunique() < 2:
        return None
    frac_pos, mean_pred = calibration_curve(pred_df['y_any_leak'], pred_df['p_any_leak'], n_bins=10, strategy='quantile')
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(mean_pred, frac_pos, marker='o')
    ax.plot([0, 1], [0, 1], linestyle='--')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed leak frequency')
    ax.set_title(title)
    fig.tight_layout()
    return fig



def make_probability_histogram_fig(pred_df: pd.DataFrame, title: str):
    if pred_df.empty or pred_df['y_any_leak'].nunique() < 2:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(pred_df.loc[pred_df['y_any_leak'] == 0, 'p_any_leak'], bins=20, alpha=0.7, label='no leak')
    ax.hist(pred_df.loc[pred_df['y_any_leak'] == 1, 'p_any_leak'], bins=20, alpha=0.7, label='leak')
    ax.set_xlabel('Calibrated p_any_leak')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig



def make_feature_importance_fig(importance_df: pd.DataFrame, title: str, top_n: int = 25):
    if importance_df.empty:
        return None
    top = importance_df.head(top_n).sort_values('importance', ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(6, int(top_n * 0.28))))
    ax.barh(top['feature'], top['importance'])
    ax.set_xlabel('Importance')
    ax.set_title(title)
    fig.tight_layout()
    return fig



def make_threshold_tradeoff_fig(threshold_table: pd.DataFrame, title: str):
    if threshold_table.empty:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(threshold_table['threshold'], threshold_table['precision'], label='precision')
    ax.plot(threshold_table['threshold'], threshold_table['recall'], label='recall')
    ax.plot(threshold_table['threshold'], threshold_table['accepted_fraction'], label='accepted_fraction')
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Metric value')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
# ========================
# Stage 2 pipeline
# ========================

stage2_cfg = PIPELINE_CFG
with open(stage2_cfg.config_path, 'r') as f:
    stage1_cfg = Stage1Config(**json.load(f))

full_df = load_battledim_merged_csv(stage2_cfg.data_path, timestamp_col=stage2_cfg.timestamp_col)
column_groups = infer_column_groups(full_df, include_flow3_context=stage2_cfg.include_flow3_context)

full_df_2018 = full_df[full_df.index.year == 2018].copy()
full_df_2019 = full_df[full_df.index.year == 2019].copy()
stage1_fit_end = choose_stage1_fit_end_from_healthy_block(
    full_df_2018,
    fit_frac=stage2_cfg.stage1_fit_healthy_frac,
    pipe_cols=column_groups['pipe_cols'],
)

stage1_train_df = full_df.loc[:stage1_fit_end].copy()
post_fit_df = full_df.loc[full_df.index > stage1_fit_end].copy()
val2018_df = post_fit_df[post_fit_df.index.year == 2018].copy()
test2019_df = post_fit_df[post_fit_df.index.year == 2019].copy()

stage1_train_exclusions = build_known_intervals_from_leak_columns(
    stage1_train_df,
    pad_before='30min',
    pad_after='30min',
    min_active_samples=1,
)

stage2_detector = BattLeDIMStage1(stage1_cfg)
stage2_detector.fit(stage1_train_df, known_intervals=stage1_train_exclusions)
results_postfit, sensor_detail_postfit, events_postfit = stage2_detector.detect(post_fit_df)
triggers_postfit = stage2_detector.build_stage2_triggers(results_postfit)
triggers_postfit = triggers_postfit.copy()
triggers_postfit['timestamp'] = pd.to_datetime(triggers_postfit['timestamp'])
triggers_postfit = triggers_postfit.sort_values('timestamp').reset_index(drop=True)

results_2018 = results_postfit.loc[val2018_df.index].copy()
results_2019 = results_postfit.loc[test2019_df.index].copy()

truth_2018 = get_any_real_leak_mask(val2018_df, column_groups['pipe_cols']).astype(int)
truth_2019 = get_any_real_leak_mask(test2019_df, column_groups['pipe_cols']).astype(int)

triggers_2018 = triggers_postfit[(triggers_postfit['timestamp'] >= val2018_df.index.min()) & (triggers_postfit['timestamp'] <= val2018_df.index.max())].copy()
triggers_2019 = triggers_postfit[(triggers_postfit['timestamp'] >= test2019_df.index.min()) & (triggers_postfit['timestamp'] <= test2019_df.index.max())].copy()

stage1_metrics_2018 = evaluate_stage1_results(results_2018, truth_2018, triggers=triggers_2018)
stage1_metrics_2019 = evaluate_stage1_results(results_2019, truth_2019, triggers=triggers_2019)

save_series(stage1_metrics_2018, 'stage1_metrics_2018_postfit', description='Stage 1 metrics on late-2018 post-fit data.')
save_series(stage1_metrics_2019, 'stage1_metrics_2019_holdout', description='Stage 1 metrics on 2019 holdout data.')
save_frame(events_postfit, 'stage1_events_postfit', index=False, description='Stage 1 event summary on all post-fit data.', max_md_rows=40)
save_frame(triggers_postfit, 'stage1_triggers_postfit', index=False, description='Stage 1 trigger packets used to wake Stage 2.', max_md_rows=80)

# Full post-fit Stage 1 outputs can be large, but they are useful for later report work.
results_postfit.to_csv(TABLE_DIR / 'stage1_results_postfit_full.csv')
add_artifact('table', 'stage1_results_postfit_full', TABLE_DIR / 'stage1_results_postfit_full.csv', 'Full Stage 1 results for all post-fit timestamps.')
sensor_detail_postfit.to_csv(TABLE_DIR / 'stage1_sensor_detail_postfit_full.csv')
add_artifact('table', 'stage1_sensor_detail_postfit_full', TABLE_DIR / 'stage1_sensor_detail_postfit_full.csv', 'Full per-sensor Stage 1 evidence for all post-fit timestamps.')


def build_patched_stage2_packet_splits(post_fit_df: pd.DataFrame, results: pd.DataFrame, triggers_all: pd.DataFrame, groups: dict, cfg: Stage2RunConfig):
    dev_df = post_fit_df[post_fit_df.index.year == 2018].copy()
    test_df_local = post_fit_df[post_fit_df.index.year == 2019].copy()
    y_any_dev = get_any_real_leak_mask(dev_df, groups['pipe_cols']).astype(int)

    actual_dev = triggers_all[(triggers_all['timestamp'] >= dev_df.index.min()) & (triggers_all['timestamp'] <= dev_df.index.max())].copy()
    actual_dev['timestamp'] = pd.to_datetime(actual_dev['timestamp'])
    actual_dev = actual_dev.sort_values('timestamp').reset_index(drop=True)
    actual_dev['sampled_negative'] = 0
    actual_dev['pseudo_negative_kind'] = 'none'
    actual_dev['y_any_leak'] = y_any_dev.reindex(pd.DatetimeIndex(actual_dev['timestamp'])).astype(int).values

    actual_pos = actual_dev[actual_dev['y_any_leak'] == 1].copy()
    actual_neg = actual_dev[actual_dev['y_any_leak'] == 0].copy()

    pos_train, pos_cal, pos_val = split_df_threeway(actual_pos, fracs=(0.60, 0.20, 0.20))
    neg_train, neg_cal, neg_val = split_small_temporal(actual_neg)

    actual_test = triggers_all[(triggers_all['timestamp'] >= test_df_local.index.min()) & (triggers_all['timestamp'] <= test_df_local.index.max())].copy()
    actual_test['timestamp'] = pd.to_datetime(actual_test['timestamp'])
    actual_test = actual_test.sort_values('timestamp').reset_index(drop=True)
    actual_test['sampled_negative'] = 0
    actual_test['pseudo_negative_kind'] = 'none'

    actual_dev_ts = pd.DatetimeIndex(pd.to_datetime(actual_dev['timestamp']))
    healthy_non_trigger = dev_df.index[(y_any_dev == 0).values].difference(actual_dev_ts)

    score_for_peaks = results.loc[healthy_non_trigger, 'score_final'].fillna(0.0) if len(healthy_non_trigger) else pd.Series(dtype=float)
    peak_target_total = min(len(healthy_non_trigger), max(24, int(round((len(pos_train) + len(pos_cal) + len(pos_val)) * cfg.hard_peak_negative_ratio_per_positive))))
    hard_peak_idx = select_peak_negative_timestamps(
        score_for_peaks,
        n_samples=peak_target_total,
        step_minutes=cfg.step_minutes,
        min_gap_steps=12,
        peak_radius=3,
    )
    hard_peak_train_idx, hard_peak_cal_idx, hard_peak_val_idx = split_index_threeway(hard_peak_idx)

    remaining_healthy = healthy_non_trigger.difference(hard_peak_idx)
    rem_train_df, rem_tmp_df = chronological_split_frame(pd.DataFrame({'timestamp': remaining_healthy}), 0.60)
    rem_cal_df, rem_val_df = chronological_split_frame(rem_tmp_df, 0.50)
    rem_train_idx = pd.DatetimeIndex(pd.to_datetime(rem_train_df['timestamp']))
    rem_cal_idx = pd.DatetimeIndex(pd.to_datetime(rem_cal_df['timestamp']))
    rem_val_idx = pd.DatetimeIndex(pd.to_datetime(rem_val_df['timestamp']))

    easy_train_target = int(round(len(pos_train) * cfg.easy_negative_ratio_per_positive))
    easy_cal_target = int(round(len(pos_cal) * cfg.easy_negative_ratio_per_positive))
    easy_val_target = int(round(len(pos_val) * cfg.easy_negative_ratio_per_positive))

    easy_train_idx = stratified_sample_timestamps(rem_train_idx, easy_train_target, seed=cfg.random_state + 10)
    easy_cal_idx = stratified_sample_timestamps(rem_cal_idx, easy_cal_target, seed=cfg.random_state + 11)
    easy_val_idx = stratified_sample_timestamps(rem_val_idx, easy_val_target, seed=cfg.random_state + 12)

    train_packets = pd.concat([
        pos_train.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        neg_train.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        make_pseudo_trigger_frame(hard_peak_train_idx, reason='pseudo_hard_negative', kind='pseudo_hard_negative'),
        make_pseudo_trigger_frame(easy_train_idx, reason='sampled_negative', kind='sampled_negative'),
    ], ignore_index=True).sort_values('timestamp').reset_index(drop=True)

    cal_packets = pd.concat([
        pos_cal.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        neg_cal.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        make_pseudo_trigger_frame(hard_peak_cal_idx, reason='pseudo_hard_negative', kind='pseudo_hard_negative'),
        make_pseudo_trigger_frame(easy_cal_idx, reason='sampled_negative', kind='sampled_negative'),
    ], ignore_index=True).sort_values('timestamp').reset_index(drop=True)

    val_packets = pd.concat([
        pos_val.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        neg_val.assign(sampled_negative=0, pseudo_negative_kind='none').drop(columns=['y_any_leak'], errors='ignore'),
        make_pseudo_trigger_frame(hard_peak_val_idx, reason='pseudo_hard_negative', kind='pseudo_hard_negative'),
        make_pseudo_trigger_frame(easy_val_idx, reason='sampled_negative', kind='sampled_negative'),
    ], ignore_index=True).sort_values('timestamp').reset_index(drop=True)

    info = {
        'actual_dev_triggers': int(len(actual_dev)),
        'actual_dev_positive_triggers': int(len(actual_pos)),
        'actual_dev_negative_triggers': int(len(actual_neg)),
        'pseudo_hard_negative_total': int(len(hard_peak_idx)),
        'healthy_non_trigger_pool': int(len(healthy_non_trigger)),
        'easy_negative_total': int(len(easy_train_idx) + len(easy_cal_idx) + len(easy_val_idx)),
    }
    return train_packets, cal_packets, val_packets, actual_test, info


train_packets, cal_packets, val_packets, test_packets, packet_info = build_patched_stage2_packet_splits(
    post_fit_df=post_fit_df,
    results=results_postfit,
    triggers_all=triggers_postfit,
    groups=column_groups,
    cfg=stage2_cfg,
)

all_packet_triggers = pd.concat([train_packets, cal_packets, val_packets, test_packets], ignore_index=True).sort_values('timestamp').reset_index(drop=True)
feature_builder = Stage2FeatureBuilder(
    raw_df=post_fit_df,
    results=results_postfit,
    sensor_detail=sensor_detail_postfit,
    all_triggers=all_packet_triggers,
    groups=column_groups,
    config=stage2_cfg,
)

X_train, meta_train, y_train = feature_builder.build_dataset(train_packets, split_name='stage2_train')
X_cal, meta_cal, y_cal = feature_builder.build_dataset(cal_packets, split_name='stage2_cal')
X_val, meta_val, y_val = feature_builder.build_dataset(val_packets, split_name='stage2_val')
X_test, meta_test, y_test = feature_builder.build_dataset(test_packets, split_name='test_2019')

X_train, X_cal, X_val, X_test = align_feature_frames(X_train, X_cal, X_val, X_test)

binary_cols = select_binary_feature_columns(X_train.columns)
loc_cols = select_localisation_feature_columns(X_train.columns)

Xb_train = X_train[binary_cols].copy()
Xb_cal = X_cal[binary_cols].copy()
Xb_val = X_val[binary_cols].copy()
Xb_test = X_test[binary_cols].copy()

Xloc_train_full = X_train[loc_cols].copy()
Xloc_val_full = X_val[loc_cols].copy()
Xloc_test_full = X_test[loc_cols].copy()

packet_summary_df = pd.DataFrame([
    packet_summary(meta_train, y_train, 'stage2_train'),
    packet_summary(meta_cal, y_cal, 'stage2_cal'),
    packet_summary(meta_val, y_val, 'stage2_val'),
    packet_summary(meta_test, y_test, 'test_2019'),
])
save_frame(packet_summary_df, 'stage2_packet_summary', index=False, description='Packet counts and class balance by split.', max_md_rows=20)
save_json(packet_info, 'stage2_packet_source_counts', description='How Stage 2 trigger packets were constructed.')

binary_source_weight = np.ones(len(meta_train), dtype=float)
kind_train = meta_train.get('pseudo_negative_kind', pd.Series('none', index=meta_train.index)).fillna('none').astype(str)
binary_source_weight[kind_train.eq('sampled_negative').to_numpy()] = stage2_cfg.sampled_negative_weight
binary_source_weight[kind_train.eq('pseudo_hard_negative').to_numpy()] = stage2_cfg.pseudo_hard_negative_weight
binary_source_weight[((meta_train['sampled_negative'].astype(int) == 0) & (y_train['y_any_leak'] == 0)).to_numpy()] = stage2_cfg.real_hard_negative_weight
binary_class_weight = compute_sample_weight(class_weight='balanced', y=y_train['y_any_leak'].to_numpy(dtype=int))
sample_weight_binary = binary_source_weight * binary_class_weight

binary_model, binary_backend = fit_binary_model(
    Xb_train,
    y_train['y_any_leak'].to_numpy(dtype=int),
    sample_weight=sample_weight_binary,
    random_state=stage2_cfg.random_state,
)

p_train_raw = binary_model.predict_proba(Xb_train)[:, 1]
p_cal_raw = binary_model.predict_proba(Xb_cal)[:, 1]
p_val_raw = binary_model.predict_proba(Xb_val)[:, 1]
p_test_raw = binary_model.predict_proba(Xb_test)[:, 1] if len(Xb_test) else np.array([])

binary_calibrator = ProbabilityCalibrator(stage2_cfg.random_state).fit(p_cal_raw, y_cal['y_any_leak'].to_numpy(dtype=int))
p_train = binary_calibrator.predict(p_train_raw)
p_cal = binary_calibrator.predict(p_cal_raw)
p_val = binary_calibrator.predict(p_val_raw)
p_test = binary_calibrator.predict(p_test_raw) if len(p_test_raw) else np.array([])

threshold_info, threshold_table = direct_threshold_search(y_val['y_any_leak'].to_numpy(dtype=int), p_val, min_recall=stage2_cfg.binary_min_recall)
stage2_threshold = float(threshold_info['threshold'])

loc_train_mask = (meta_train['sampled_negative'].astype(int) == 0) & (y_train['y_any_leak'] == 1)
loc_val_mask = (meta_val['sampled_negative'].astype(int) == 0) & (y_val['y_any_leak'] == 1)
loc_test_mask = (meta_test['sampled_negative'].astype(int) == 0) & (y_test['y_any_leak'] == 1)

Xloc_train = Xloc_train_full.loc[loc_train_mask].copy()
yloc_train = y_train.loc[loc_train_mask, column_groups['pipe_cols']].copy()
pipe_signature = yloc_train.astype(int).astype(str).agg(''.join, axis=1) if not yloc_train.empty else pd.Series(dtype=str)
sig_count = pipe_signature.value_counts() if not pipe_signature.empty else pd.Series(dtype=int)
loc_weight = ((1.0 / pipe_signature.map(sig_count)).astype(float).to_numpy() if not pipe_signature.empty else np.array([]))
if len(loc_weight):
    loc_weight = loc_weight / np.mean(loc_weight)

pipe_models = {}
pipe_prob_val_full = pd.DataFrame(index=y_val.index)
pipe_prob_test_full = pd.DataFrame(index=y_test.index)
for pipe_col in column_groups['pipe_cols']:
    y_pipe_train = yloc_train[pipe_col].to_numpy(dtype=int) if not yloc_train.empty else np.array([])
    if len(y_pipe_train) and len(np.unique(y_pipe_train)) >= 2:
        per_pipe_weight = loc_weight * compute_sample_weight(class_weight='balanced', y=y_pipe_train)
    else:
        per_pipe_weight = loc_weight if len(loc_weight) else None
    pipe_model, _ = fit_binary_model(
        Xloc_train,
        y_pipe_train if len(y_pipe_train) else np.array([0]),
        sample_weight=per_pipe_weight,
        random_state=stage2_cfg.random_state,
    ) if len(Xloc_train) else (ConstantBinaryModel(0.0), 'constant')
    pipe_models[pipe_col] = pipe_model
    pipe_prob_val_full[pipe_col] = pipe_model.predict_proba(Xloc_val_full)[:, 1] if len(Xloc_val_full) else np.array([])
    pipe_prob_test_full[pipe_col] = pipe_model.predict_proba(Xloc_test_full)[:, 1] if len(Xloc_test_full) else np.array([])

reg_pred_val = None
reg_pred_test = None
reg_backend = None
if stage2_cfg.enable_regression_head and {'y_leak_mag_total', 'y_leak_count'}.issubset(y_train.columns) and len(Xloc_train):
    reg_targets = ['y_leak_mag_total', 'y_leak_count']
    reg_model, reg_backend = fit_regression_model(
        Xloc_train,
        y_train.loc[loc_train_mask, reg_targets],
        sample_weight=loc_weight if len(loc_weight) else None,
        random_state=stage2_cfg.random_state,
    )
    reg_pred_val = pd.DataFrame(reg_model.predict(Xloc_val_full), columns=['pred_leak_mag_total', 'pred_leak_count'])
    reg_pred_test = pd.DataFrame(reg_model.predict(Xloc_test_full), columns=['pred_leak_mag_total', 'pred_leak_count']) if len(Xloc_test_full) else None

pred_val = build_prediction_frame(meta_val, y_val, p_raw=p_val_raw, p_cal=p_val, threshold=stage2_threshold, pipe_prob_df=pipe_prob_val_full, reg_pred=reg_pred_val)
pred_test = build_prediction_frame(meta_test, y_test, p_raw=p_test_raw, p_cal=p_test, threshold=stage2_threshold, pipe_prob_df=pipe_prob_test_full, reg_pred=reg_pred_test)

binary_metrics_val_mixed = safe_binary_metrics(y_val['y_any_leak'].to_numpy(dtype=int), p_val, stage2_threshold, stage2_cfg.binary_precision_targets)
val_actual_mask = meta_val['sampled_negative'].astype(int) == 0
binary_metrics_val_actual = safe_binary_metrics(y_val.loc[val_actual_mask, 'y_any_leak'].to_numpy(dtype=int), p_val[val_actual_mask.to_numpy()], stage2_threshold, stage2_cfg.binary_precision_targets)
binary_metrics_test_actual = safe_binary_metrics(y_test['y_any_leak'].to_numpy(dtype=int), p_test, stage2_threshold, stage2_cfg.binary_precision_targets)

localisation_metrics_val = evaluate_localisation(y_val.loc[loc_val_mask, column_groups['pipe_cols']], pipe_prob_val_full.loc[loc_val_mask])
localisation_metrics_test = evaluate_localisation(y_test.loc[loc_test_mask, column_groups['pipe_cols']], pipe_prob_test_full.loc[loc_test_mask])

importance_df = feature_importance_frame(binary_model, Xb_train.columns, X_val=Xb_val, y_val=y_val['y_any_leak'].to_numpy(dtype=int), max_proxy_features=60, n_repeats=3)
group_importance_df = (importance_df.assign(group=importance_df['feature'].map(feature_group)).groupby('group', as_index=False)['importance'].sum().sort_values('importance', ascending=False)) if not importance_df.empty else pd.DataFrame(columns=['group', 'importance'])

reason_breakdown_val = reason_acceptance_breakdown(pred_val)
reason_breakdown_test = reason_acceptance_breakdown(pred_test)
operational_val = operational_binary_summary(meta_val, y_val, p_val, stage2_threshold)
operational_test = operational_binary_summary(meta_test, y_test, p_test, stage2_threshold)

sanity_flags = pd.Series({
    'stage1_fit_end': str(stage1_fit_end),
    'stage1_train_rows': int(len(stage1_train_df)),
    'late_2018_rows': int(len(val2018_df)),
    'holdout_2019_rows': int(len(test2019_df)),
    'holdout_2019_all_positive_binary': bool(truth_2019.nunique() == 1 and truth_2019.iloc[0] == 1) if len(truth_2019) else False,
    'val_actual_has_real_hard_negatives': bool(((meta_val['sampled_negative'].astype(int) == 0) & (y_val['y_any_leak'] == 0)).any()) if len(meta_val) else False,
    'binary_threshold_accepts_all_actual_val': bool(np.isfinite(operational_val.get('accept_frac_actual_triggers', np.nan)) and operational_val.get('accept_frac_actual_triggers', np.nan) >= 0.999),
    'binary_threshold': float(stage2_threshold),
    'binary_backend': binary_backend,
    'binary_calibrator': binary_calibrator.method,
    'regression_backend': reg_backend,
    'binary_feature_count': int(len(binary_cols)),
    'localisation_feature_count': int(len(loc_cols)),
})

save_series(binary_metrics_val_mixed, 'binary_metrics_val_mixed', description='Binary Stage 2 metrics on patched validation packets: actual triggers plus pseudo-negatives.')
save_series(binary_metrics_val_actual, 'binary_metrics_val_actual_only', description='Binary Stage 2 metrics on actual Stage 1 validation triggers only.')
save_series(binary_metrics_test_actual, 'binary_metrics_test_2019_actual', description='Binary Stage 2 metrics on 2019 actual triggers.')
save_series(localisation_metrics_val, 'localisation_metrics_val_actual_positive', description='Pipe localisation metrics on actual positive validation triggers.')
save_series(localisation_metrics_test, 'localisation_metrics_test_2019_actual_positive', description='Pipe localisation metrics on 2019 actual positive triggers.')
save_series(operational_val, 'operational_summary_val', description='Operational acceptance summary on validation packets.')
save_series(operational_test, 'operational_summary_test_2019', description='Operational acceptance summary on 2019 packets.')
save_series(sanity_flags, 'sanity_flags', description='Run-level sanity checks for interpretation.')
save_json(threshold_info, 'binary_threshold_choice', description='Chosen Stage 2 binary operating point.')
save_frame(threshold_table, 'binary_threshold_tradeoff_table', index=False, description='Precision / recall / acceptance vs threshold.', max_md_rows=80)
save_frame(reason_breakdown_val.reset_index(), 'reason_breakdown_val', index=False, description='Accepted vs rejected actual validation triggers by trigger reason.', max_md_rows=20)
save_frame(reason_breakdown_test.reset_index(), 'reason_breakdown_test_2019', index=False, description='Accepted vs rejected 2019 triggers by trigger reason.', max_md_rows=20)
save_frame(importance_df, 'binary_feature_importance', index=False, description='Feature importance for the binary head. Native if available, otherwise proxy permutation on the validation set.', max_md_rows=80)
save_frame(group_importance_df, 'binary_group_importance', index=False, description='Feature importance aggregated by feature family.', max_md_rows=20)
save_frame(pred_val, 'predictions_val_full', index=False, description='Full validation trigger predictions.', max_md_rows=80)
save_frame(pred_test, 'predictions_test_2019_full', index=False, description='Full 2019 trigger predictions.', max_md_rows=80)
save_frame(pred_test[['timestamp', 'reason', 'event_id', 'p_any_leak', 'pred_any_leak', 'top_1_pipe', 'top_2_pipe', 'top_3_pipe', 'top_3_pipes', 'true_pipes']].head(100), 'predictions_test_2019_report_view', index=False, description='Compact 2019 trigger table for reports.', max_md_rows=100)

metadata_obj = {
    'run_dir': str(RUN_DIR.resolve()),
    'pipeline_config': stage2_cfg,
    'resolved_data_path': str(stage2_cfg.data_path),
    'resolved_config_path': str(stage2_cfg.config_path),
    'packet_source_counts': packet_info,
    'threshold_choice': threshold_info,
    'binary_backend': binary_backend,
    'binary_calibrator': binary_calibrator.method,
    'regression_backend': reg_backend,
}
save_json(metadata_obj, 'run_metadata', description='Top-level metadata for this notebook run.')

fig_pr = make_pr_curve_fig(pred_val, 'Patched Stage 2 validation PR curve')
if fig_pr is not None:
    save_figure(fig_pr, 'validation_pr_curve', description='Precision-recall curve on patched validation packets.')
fig_cal = make_calibration_fig(pred_val, 'Patched Stage 2 validation calibration')
if fig_cal is not None:
    save_figure(fig_cal, 'validation_calibration_curve', description='Calibration curve on patched validation packets.')
fig_hist = make_probability_histogram_fig(pred_val, 'Patched Stage 2 validation probability histogram')
if fig_hist is not None:
    save_figure(fig_hist, 'validation_probability_histogram', description='Histogram of Stage 2 validation probabilities.')
fig_imp = make_feature_importance_fig(importance_df, 'Patched Stage 2 binary head feature importance', top_n=25)
if fig_imp is not None:
    save_figure(fig_imp, 'binary_feature_importance_top25', description='Top binary-head features.')
fig_thr = make_threshold_tradeoff_fig(threshold_table, 'Binary threshold tradeoff')
if fig_thr is not None:
    save_figure(fig_thr, 'binary_threshold_tradeoff', description='Precision/recall/acceptance versus decision threshold.')

# Report-ready narrative
key_takeaways = []
if sanity_flags['holdout_2019_all_positive_binary']:
    key_takeaways.append('2019 binary labels are all positive in this merged table, so 2019 cannot measure false-alarm rejection for the binary head.')
if sanity_flags['val_actual_has_real_hard_negatives']:
    key_takeaways.append('Validation actual triggers include real Stage 1 hard negatives, so the binary confirmation head is being tested on at least some genuine false-trigger cases.')
else:
    key_takeaways.append('Validation actual triggers do not contain real Stage 1 hard negatives; binary confirmation is still judged mostly with pseudo-negatives.')
if np.isfinite(operational_val.get('accept_frac_actual_triggers', np.nan)):
    key_takeaways.append(f"Stage 2 accepts {100 * operational_val['accept_frac_actual_triggers']:.1f}% of actual validation triggers at the chosen threshold.")
if np.isfinite(operational_val.get('accept_frac_real_hard_negatives', np.nan)):
    key_takeaways.append(f"Stage 2 accepts {100 * operational_val['accept_frac_real_hard_negatives']:.1f}% of real hard-negative validation triggers.")
if np.isfinite(localisation_metrics_test.get('top3_hit_rate', np.nan)):
    key_takeaways.append(f"2019 localisation top-3 hit rate is {100 * localisation_metrics_test['top3_hit_rate']:.1f}%.")
if np.isfinite(localisation_metrics_test.get('top1_hit_rate', np.nan)):
    key_takeaways.append(f"2019 localisation top-1 hit rate is {100 * localisation_metrics_test['top1_hit_rate']:.1f}%.")

report_split = _frame_to_markdown(packet_summary_df, index=False, max_rows=20)
report_stage1_2018 = _frame_to_markdown(stage1_metrics_2018.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=50)
report_stage1_2019 = _frame_to_markdown(stage1_metrics_2019.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=50)
report_binary_val = _frame_to_markdown(binary_metrics_val_actual.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=50)
report_binary_test = _frame_to_markdown(binary_metrics_test_actual.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=50)
report_loc_val = _frame_to_markdown(localisation_metrics_val.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=20)
report_loc_test = _frame_to_markdown(localisation_metrics_test.rename('value').reset_index().rename(columns={'index': 'metric'}), index=False, max_rows=20)
report_group_imp = _frame_to_markdown(group_importance_df.head(12), index=False, max_rows=12)
report_pred = _frame_to_markdown(pred_test[['timestamp', 'reason', 'p_any_leak', 'top_1_pipe', 'top_2_pipe', 'top_3_pipe', 'true_pipes']].head(20), index=False, max_rows=20)

add_report_section('Run context', f"Run folder: `{RUN_DIR.resolve()}`\n\nResolved data path: `{stage2_cfg.data_path}`\n\nResolved config path: `{stage2_cfg.config_path}`")
add_report_section('Key takeaways', '\n'.join([f'- {x}' for x in key_takeaways]))
add_report_section('Packet split summary', report_split)
add_report_section('Stage 1 metrics on late 2018', report_stage1_2018)
add_report_section('Stage 1 metrics on 2019', report_stage1_2019)
add_report_section('Binary Stage 2 metrics on actual validation triggers', report_binary_val)
add_report_section('Binary Stage 2 metrics on 2019 actual triggers', report_binary_test)
add_report_section('Localisation metrics on validation positives', report_loc_val)
add_report_section('Localisation metrics on 2019 positives', report_loc_test)
add_report_section('Top binary feature groups', report_group_imp)
add_report_section('Example 2019 predictions', report_pred)

finalize_run_summary()

print('\nRun folder:', RUN_DIR.resolve())
print('\nStage 1 metrics on late 2018:')
display(stage1_metrics_2018.to_frame('value'))
print('\nStage 1 metrics on 2019:')
display(stage1_metrics_2019.to_frame('value'))
print('\nStage 2 packet summary:')
display(packet_summary_df)
print('\nBinary metrics on actual validation triggers:')
display(binary_metrics_val_actual.to_frame('value'))
print('\nBinary metrics on 2019 actual triggers:')
display(binary_metrics_test_actual.to_frame('value'))
print('\nLocalisation metrics on 2019 positives:')
display(localisation_metrics_test.to_frame('value'))
print('\nTrigger reason breakdown on 2019:')
display(reason_breakdown_test)
print('\nTop feature groups:')
display(group_importance_df.head(12))
print('\nExample 2019 predictions:')
display(pred_test[['timestamp', 'reason', 'event_id', 'p_any_leak', 'pred_any_leak', 'top_1_pipe', 'top_2_pipe', 'top_3_pipe', 'top_3_pipes', 'true_pipes']].head(25))
print('\nArtifacts written to:', RUN_DIR.resolve())